# Video Scene Analysis — Full Object & Event Detection Demo

This notebook demonstrates the end-to-end pipeline for analysing dashcam / CCTV
footage using **Gemini 2.5 Pro** via **LiteLLM**. Unlike the crash-only variant,
this version instructs the model to **describe every visible object, road user,
environmental feature, and event** in each frame — not just collisions.

**Pipeline steps:**
1. Configure LiteLLM connection
2. Define extended Pydantic schemas for full scene description
3. Load & validate a sample dashcam video
4. Encode the video to base64 and build the multimodal payload
5. Send the request to Gemini 2.5 Pro for scene-level analysis
6. Parse the structured JSON response into Pydantic models
7. Display and explore the results

## 1. Setup & Configuration

In [ ]:
import litellm
import json
import base64
import logging
from pathlib import Path
from IPython.display import display, JSON, Video

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
# ── LiteLLM credentials ─────────────────────────────────────────────
# Replace with your own credentials or load from environment variables.

LITELLM_API_KEY = "sk-jijtUNjqiqtTr1DSQUeKMQ"
LITELLM_BASE_URL = "https://genai-lounge-nx-litellm-uat-emea.zurich.com"

litellm.api_key = LITELLM_API_KEY
litellm.api_base = LITELLM_BASE_URL
litellm.suppress_debug_info = True

# Model identifier — routed through LiteLLM to the Gemini 2.5 Pro deployment
MODEL_NAME = "gemini/gemini-2.5-pro"

print(f"LiteLLM configured — base: {LITELLM_BASE_URL}")
print(f"Model: {MODEL_NAME}")

## 2. Extended Pydantic Schemas for Scene Description

These schemas replace the crash-only `CrashEvent` with a richer structure
that captures **all** detected objects, road users, environmental context,
and any notable events (including but not limited to collisions).

In [ ]:
# ── Import scene-description schemas from the project ─────────────

import sys
from pathlib import Path

# Ensure the project root is on sys.path for local imports
sys.path.insert(0, str(Path(".").resolve().parent))


from video_analysis_poc.video_llm.schema.video import (
    DetectedObject,
    CrashInvolvedParty,
    WitnessedCrashDetail,
    SceneEvent,
    EnvironmentContext,
    VideoSceneAnalysisOutput,
)
print("Imported scene-description schemas from video_analysis_poc.video_llm.schema.video")

## 3. Load & Validate the Sample Video

In [ ]:
# ── Point this to your local dashcam / CCTV sample ─────────────────
VIDEO_PATH = Path("C:\\Users\\AMIR.RAHIMI\\Downloads\\Dashcam\\Inbound_CHO_Dashcam_20-50-780086.mp4")

SUPPORTED_FORMATS = {"mp4", "avi", "mov", "mkv", "webm"}
MAX_VIDEO_SIZE_MB = 100.0

VIDEO_MIME_TYPES = {
    "mp4": "video/mp4",
    "avi": "video/x-msvideo",
    "mov": "video/quicktime",
    "mkv": "video/x-matroska",
    "webm": "video/webm",
}

In [ ]:
def validate_video(path: Path) -> None:
    """Validate that the video exists, has a supported format, and is within size limits."""
    if not path.exists():
        raise FileNotFoundError(f"Video file not found: {path}")

    ext = path.suffix.lstrip(".").lower()
    if ext not in SUPPORTED_FORMATS:
        raise ValueError(f"Unsupported format '.{ext}'. Supported: {SUPPORTED_FORMATS}")

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_VIDEO_SIZE_MB:
        raise ValueError(f"Video is {size_mb:.1f} MB, exceeding limit of {MAX_VIDEO_SIZE_MB} MB.")

    print(f"Video validated: {path.name}  ({size_mb:.2f} MB, format: .{ext})")


validate_video(VIDEO_PATH)

In [ ]:
# ── Preview the video inline (works for .mp4 in JupyterLab) ────────
try:
    display(Video(str(VIDEO_PATH), embed=True, width=640))
except Exception as e:
    print(f"Inline preview not available: {e}")
    print(f"Open the file manually: {VIDEO_PATH.resolve()}")

## 4. Encode Video & Build Multimodal Payload

In [ ]:
def encode_video_to_base64(path: Path) -> str:
    """Read a video file and return its base64-encoded string."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def get_mime_type(path: Path) -> str:
    ext = path.suffix.lstrip(".").lower()
    return VIDEO_MIME_TYPES.get(ext, "video/mp4")


logger.info("Encoding video to base64 …")
video_b64 = encode_video_to_base64(VIDEO_PATH)
video_mime = get_mime_type(VIDEO_PATH)

print(f"Encoded: {len(video_b64) / (1024*1024):.2f} MB base64 string  (MIME: {video_mime})")

In [ ]:
# ── Build the multimodal content parts ──────────────────────────────
CAMERA_TYPE = "dashcam_front"

video_content_parts = [
    {
        "type": "text",
        "text": f"[{CAMERA_TYPE}] Video file: {VIDEO_PATH.name}",
    },
    {
        "type": "image_url",
        "image_url": {
            "url": f"data:{video_mime};base64,{video_b64}",
        },
    },
]

print(f"Built {len(video_content_parts)} content parts for the multimodal payload.")

## 5. Craft the Prompt & Send to Gemini 2.5 Pro

The system and task prompts have been rewritten to request **comprehensive
scene understanding** rather than crash-only detection.

In [ ]:
# ── System prompt — general scene understanding ────────────────────
SYSTEM_PROMPT = (
    "You are an expert multimodal video analyst specialising in road scene "
    "understanding from dashcam and CCTV footage. Your task is to provide a "
    "comprehensive, structured description of EVERYTHING visible in the video: "
    "all vehicles, pedestrians, cyclists, road infrastructure, signage, lane "
    "markings, buildings, vegetation, weather and lighting conditions, and any "
    "notable events (lane changes, braking, turns, pedestrian crossings, etc.). "
    "Pay special attention to any collisions or near-miss events — whether the "
    "dashcam vehicle is directly involved OR merely witnesses a crash between "
    "other road users. For every collision or near-miss, provide a detailed "
    "forensic-level breakdown: impact type, each involved party's description "
    "and actions before/after, probable cause, fault indicators, sequence of "
    "events, post-crash actions, and estimated speeds. Be thorough and precise "
    "with timestamps. Return your findings as a structured JSON object."
)

# ── Task prompt — full scene description schema ────────────────────
TASK_PROMPT = """\
Analyse the attached video and provide a complete scene description.
Return a JSON object with the following schema:

{
  "environment": {
    "weather": "<clear | rain | fog | snow | overcast | ...>",
    "lighting": "<daylight | dusk | dawn | night | artificial | ...>",
    "road_type": "<motorway | urban | suburban | rural | roundabout | intersection | ...>",
    "road_surface": "<dry | wet | icy | gravel | ...>",
    "traffic_density": "<empty | light | moderate | heavy | congested>",
    "additional_notes": "<any other relevant environmental observations>"
  },
  "detected_objects": [
    {
      "object_type": "<car | truck | bus | motorcycle | bicycle | pedestrian | traffic_sign | traffic_light | lane_marking | road_barrier | building | tree | animal | debris | other>",
      "description": "<brief description including colour, size, behaviour, or state>",
      "first_seen_seconds": <float or null>,
      "last_seen_seconds": <float or null>,
      "position_in_frame": "<centre | left_lane | right_lane | left_shoulder | right_shoulder | far_background | ...>",
      "confidence": <float 0-1>
    }
  ],
  "events": [
    {
      "event_type": "<collision | near_miss | hard_braking | lane_change | pedestrian_crossing | traffic_light_change | vehicle_overtake | vehicle_turning | emergency_vehicle | road_obstruction | other>",
      "timestamp_seconds": <float>,
      "timestamp_display": "<HH:MM:SS>",
      "description": "<what happened>",
      "involved_objects": ["<object_type>", ...],
      "severity": "<low | medium | high | null>",
      "confidence": <float 0-1>,
      "witnessed_crash_detail": {
        "impact_type": "<rear_end | head_on | side_swipe | t_bone | rollover | run_off_road | pedestrian_strike | cyclist_strike | multi_vehicle_pileup | none_near_miss | other>",
        "involved_parties": [
          {
            "road_user_type": "<car | truck | bus | motorcycle | bicycle | pedestrian | other>",
            "description": "<colour, make/model if visible, lane, direction>",
            "action_before_event": "<what they were doing just before>",
            "action_after_event": "<what they did after>",
            "apparent_damage": "<visible damage or 'none visible'>"
          }
        ],
        "probable_cause": "<primary cause assessment>",
        "fault_indicators": ["<observable cue suggesting fault>", ...],
        "sequence_of_events": "<step-by-step narrative of how the crash unfolded>",
        "post_crash_actions": "<what happened after: vehicles stopping, emergency response, debris, etc.>",
        "estimated_speed_range": "<low (< 20 mph) | moderate (20-40 mph) | high (> 40 mph) | null>",
        "dashcam_vehicle_involved": <true | false>
      }
    }
  ],
  "summary": "<overall narrative summary of everything observed in the footage>"
}

Instructions:
- List EVERY distinct object you can identify, not just vehicles.
- Include static infrastructure: signs, traffic lights, barriers, road markings.
- Include environmental features: buildings, trees, weather cues.
- For events, capture ALL notable actions — not just collisions.
  Examples: a car changing lanes, a pedestrian crossing, a traffic light
  turning red, a vehicle braking hard, an overtake, a turn at a junction.
- CRASH / NEAR-MISS EVENTS require special attention:
  * Set event_type to 'collision' or 'near_miss' as appropriate.
  * ALWAYS populate the 'witnessed_crash_detail' object for these events.
  * This applies whether the dashcam vehicle is directly involved in the
    crash OR is merely a bystander witnessing a crash between other vehicles.
  * Describe each involved party separately with their actions before and
    after the event, visible damage, and any other observable details.
  * Assess probable cause and list all fault indicators you can observe.
  * Provide a step-by-step sequence-of-events narrative.
  * Note any post-crash actions: vehicles stopping, hazard lights, emergency
    services, debris on road, other vehicles slowing down.
  * Set dashcam_vehicle_involved to true only if the recording vehicle
    itself was part of the collision; false if it was a witness.
- For all NON-crash events, set 'witnessed_crash_detail' to null.
- Provide timestamps relative to the start of the video.
- Be conservative with confidence scores.
- If the same object persists across many seconds, report it once with
  first_seen_seconds and last_seen_seconds.
- Return ONLY the JSON object, no markdown fences or commentary.
"""

print("System prompt length:", len(SYSTEM_PROMPT), "chars")
print("Task prompt length:  ", len(TASK_PROMPT), "chars")

In [ ]:
# ── Assemble the full messages payload ────────────────────────────
user_content = video_content_parts + [
    {"type": "text", "text": TASK_PROMPT}
]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_content},
]

print(f"Messages payload assembled — {len(messages)} messages, "
      f"{len(user_content)} user content parts.")

In [ ]:
# ── Call Gemini 2.5 Pro via LiteLLM ─────────────────────────────────
import time as _time

TEMPERATURE = 0.0

logger.info("Sending multimodal request to %s …", MODEL_NAME)

_start = _time.perf_counter()

raw_response = await litellm.acompletion(
    model=MODEL_NAME,
    messages=messages,
    temperature=TEMPERATURE,
    response_format={"type": "json_object"},
    cache={"no-store": True},
)

_elapsed = _time.perf_counter() - _start
logger.info("Response received — model: %s (%.2f s)", getattr(raw_response, "model", "unknown"), _elapsed)

## 6. Parse the Response into Pydantic Models

In [ ]:
from litellm.types.utils import ModelResponse, Choices

if not isinstance(raw_response, ModelResponse):
    raise TypeError(
        f"Expected ModelResponse, got {type(raw_response).__name__}. "
        "Streaming responses are not supported in this demo."
    )

choice = raw_response.choices[0]
if not isinstance(choice, Choices):
    raise TypeError(
        f"Expected Choices, got {type(choice).__name__}. "
        "Streaming responses are not supported in this demo."
    )

chat_content = choice.message.content
if not chat_content:
    raise ValueError("LLM returned empty content.")

# Strip markdown fences if the model wrapped the JSON
clean_content = chat_content.strip()
if clean_content.startswith("```"):
    clean_content = clean_content.split("\n", 1)[1]
    if clean_content.endswith("```"):
        clean_content = clean_content[:-3].strip()

parsed = json.loads(clean_content)
print("JSON parsed successfully.")
print(f"Top-level keys: {list(parsed.keys())}")

In [ ]:
# ── Build the structured output ────────────────────────────────────
environment = (
    EnvironmentContext(**parsed["environment"])
    if "environment" in parsed and parsed["environment"]
    else None
)

detected_objects = [
    DetectedObject(**obj) for obj in parsed.get("detected_objects", [])
]

# Parse events, handling the nested witnessed_crash_detail for collisions/near-misses
events = []
for evt_raw in parsed.get("events", []):
    crash_detail_raw = evt_raw.pop("witnessed_crash_detail", None)
    crash_detail = None
    if crash_detail_raw is not None:
        parties_raw = crash_detail_raw.pop("involved_parties", [])
        parties = [CrashInvolvedParty(**p) for p in parties_raw]
        crash_detail = WitnessedCrashDetail(involved_parties=parties, **crash_detail_raw)
    events.append(SceneEvent(witnessed_crash_detail=crash_detail, **evt_raw))

analysis_output = VideoSceneAnalysisOutput(
    task_name="video_scene_analysis_demo",
    camera_type=CAMERA_TYPE,
    video_file_name=VIDEO_PATH.name,
    environment=environment,
    detected_objects=detected_objects,
    events=events,
    summary=parsed.get("summary"),
    raw_output=parsed,
)

# Summarise
n_crashes = sum(1 for e in events if e.event_type in ('collision', 'near_miss'))
n_other = len(events) - n_crashes
print(f"\nDetected {len(analysis_output.detected_objects)} object(s).")
print(f"Detected {len(events)} event(s): {n_crashes} crash/near-miss, {n_other} other.")
print(f"Summary: {analysis_output.summary}")

## 7. Explore the Results

In [ ]:
# ── Environment context ────────────────────────────────────────────
if analysis_output.environment:
    env = analysis_output.environment
    print("=" * 60)
    print("  Environment Context")
    print("=" * 60)
    print(f"  Weather:         {env.weather}")
    print(f"  Lighting:        {env.lighting}")
    print(f"  Road type:       {env.road_type}")
    print(f"  Road surface:    {env.road_surface}")
    print(f"  Traffic density: {env.traffic_density}")
    if env.additional_notes:
        print(f"  Notes:           {env.additional_notes}")
else:
    print("No environment context returned.")

In [ ]:
# ── Detected objects ───────────────────────────────────────────────
import pandas as pd

if analysis_output.detected_objects:
    df_objects = pd.DataFrame([o.model_dump() for o in analysis_output.detected_objects])
    display(
        df_objects.style
        .format({"confidence": "{:.2%}"})
        .set_caption(f"Detected Objects ({len(df_objects)} total)")
    )
else:
    print("No objects detected.")

In [ ]:
# ── Object type distribution ───────────────────────────────────────
if analysis_output.detected_objects:
    type_counts = df_objects["object_type"].value_counts()
    print("\nObject type distribution:")
    for obj_type, count in type_counts.items():
        print(f"  {obj_type:<20s} {count}")

In [ ]:
# ── Events ─────────────────────────────────────────────────────────
if analysis_output.events:
    for i, event in enumerate(analysis_output.events, 1):
        print(f"\n{'=' * 60}")
        print(f"  Event {i}: {event.event_type}")
        print(f"{'=' * 60}")
        print(f"  Timestamp:   {event.timestamp_display} ({event.timestamp_seconds:.1f}s)")
        print(f"  Severity:    {event.severity or 'N/A'}")
        print(f"  Confidence:  {event.confidence:.2%}")
        print(f"  Involved:    {', '.join(event.involved_objects) if event.involved_objects else 'N/A'}")
        print(f"  Description: {event.description}")
else:
    print("No notable events detected.")

In [ ]:
# ── Events as a DataFrame ─────────────────────────────────────────
if analysis_output.events:
    df_events = pd.DataFrame([e.model_dump() for e in analysis_output.events])
    # Convert involved_objects list to a comma-separated string for display
    df_events["involved_objects"] = df_events["involved_objects"].apply(
        lambda x: ", ".join(x) if x else ""
    )
    # Drop the nested crash detail column from the summary table
    display_cols = [c for c in df_events.columns if c != "witnessed_crash_detail"]
    display(
        df_events[display_cols].style
        .format({"confidence": "{:.2%}"})
        .set_caption(f"Scene Events ({len(df_events)} total)")
    )

In [ ]:
# ── Witnessed crash / near-miss details ────────────────────────────
crash_events = [e for e in analysis_output.events if e.witnessed_crash_detail is not None]

if crash_events:
    for i, event in enumerate(crash_events, 1):
        cd = event.witnessed_crash_detail
        assert cd is not None
        print(f"\n{'#' * 60}")
        print(f"  WITNESSED CRASH/NEAR-MISS {i} — {event.event_type.upper()}")
        print(f"{'#' * 60}")
        print(f"  Timestamp:              {event.timestamp_display} ({event.timestamp_seconds:.1f}s)")
        print(f"  Severity:               {event.severity or 'N/A'}")
        print(f"  Confidence:             {event.confidence:.2%}")
        print(f"  Impact type:            {cd.impact_type or 'N/A'}")
        print(f"  Dashcam involved:       {'Yes' if cd.dashcam_vehicle_involved else 'No (witness only)'}")
        print(f"  Probable cause:         {cd.probable_cause or 'N/A'}")
        print(f"  Estimated speed range:  {cd.estimated_speed_range or 'N/A'}")
        if cd.fault_indicators:
            print(f"  Fault indicators:       {', '.join(cd.fault_indicators)}")
        print(f"\n  Sequence of events:")
        print(f"    {cd.sequence_of_events or 'N/A'}")
        print(f"\n  Post-crash actions:")
        print(f"    {cd.post_crash_actions or 'N/A'}")
        print(f"\n  Description: {event.description}")
        assert cd is not None
        if cd.involved_parties:
            print(f"\n  Involved Parties ({len(cd.involved_parties)}):")
            for j, party in enumerate(cd.involved_parties, 1):
                print(f"    ── Party {j}: {party.road_user_type} ──")
                print(f"       Description:    {party.description}")
                print(f"       Before event:   {party.action_before_event or 'N/A'}")
                print(f"       After event:    {party.action_after_event or 'N/A'}")
                print(f"       Apparent damage: {party.apparent_damage or 'N/A'}")
else:
    print("No collisions or near-misses witnessed in this footage.")

In [ ]:
# ── Raw JSON output for inspection ─────────────────────────────────
display(JSON(analysis_output.raw_output, root="raw_model_output"))

In [ ]:
# ── Token usage summary ────────────────────────────────────────────
usage = getattr(raw_response, "usage", None)
if usage:
    print("Token Usage")
    print(f"  Prompt tokens:     {getattr(usage, 'prompt_tokens', 'N/A')}")
    print(f"  Completion tokens: {getattr(usage, 'completion_tokens', 'N/A')}")
    print(f"  Total tokens:      {getattr(usage, 'total_tokens', 'N/A')}")
else:
    print("Token usage not available in response.")

In [ ]:
# ── Save results to disk ───────────────────────────────────────────
output_path = Path("./scene_analysis_results_crash.json")
output_path.write_text(json.dumps(analysis_output.model_dump(), indent=2, default=str))
print(f"Results saved to {output_path.resolve()}")

---

### Notes

- **Schema evolution**: The `DetectedObject` and `SceneEvent` schemas are
  designed to be extensible. Add new `object_type` or `event_type` values
  as the model surfaces them — the schema uses `str` rather than `Literal`
  enums to allow for open-vocabulary detection.
- **Witnessed crash detail**: When `event_type` is `collision` or `near_miss`,
  the `witnessed_crash_detail` field is populated with forensic-level metadata:
  impact type, each involved party's description and pre/post-event actions,
  fault indicators, sequence of events, post-crash actions, and whether the
  dashcam vehicle was involved or just a bystander witness.
- **Crash-specific filtering**: To extract only crash events downstream,
  filter `events` by `event_type in ('collision', 'near_miss')` and access
  the `witnessed_crash_detail` field.
- **Langfuse integration**: In production, swap the local prompt for a
  Langfuse-managed template via `collate_prompt()`.
- **Multiple cameras**: Extend `video_content_parts` with additional
  video entries (one text label + one `image_url` per camera).
- **Backward compatibility**: The `VideoSceneAnalysisOutput` schema is a
  superset of the original `VideoAnalysisOutput`. Existing crash-event
  fields map directly to the `events` list with `event_type='collision'`.